# Step 2 — Silver Layer: Clean and Enrich Bronze Data

**Bronze** = raw, untouched, exactly what the API gave us.  
**Silver** = cleaned, typed correctly, enriched with basic derived columns.

The rule: **never modify Bronze**. Silver is built from Bronze, not a replacement for it.  
If your Silver logic has a bug, you fix the code and re-run — Bronze is still there, untouched.

```
01-data/bronze/crypto_raw.csv  (raw, never touch)
        │
        │  load → fix types → validate → enrich
        ▼
01-data/silver/crypto_clean.csv  (clean, typed, enriched)
```

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

project_root = Path('../../').resolve()
bronze_path = project_root / '01-data' / 'bronze' / 'crypto_raw.csv'
silver_path = project_root / '01-data' / 'silver' / 'crypto_clean.csv'

print('Libraries loaded OK')

Libraries loaded OK


## Step 1 — Load Bronze and check raw types

When pandas reads a CSV, it guesses types. Numbers usually come through fine.  
But timestamps come in as plain strings — we need to fix that.

In [2]:
bronze = pd.read_csv(bronze_path)

print(f'Bronze shape: {bronze.shape}')
print()
print('Data types as-read from CSV:')
print(bronze.dtypes)

Bronze shape: (10, 40)

Data types as-read from CSV:
id                                      int64
name                                      str
symbol                                    str
slug                                      str
infinite_supply                          bool
circulating_supply                    float64
total_supply                          float64
max_supply                            float64
date_added                                str
num_market_pairs                        int64
cmc_rank                                int64
last_updated                              str
tvl_ratio                             float64
platform                              float64
self_reported_circulating_supply      float64
self_reported_market_cap              float64
minted_market_cap                     float64
tags                                      str
quote_USD_price                       float64
quote_USD_volume_24h                  float64
quote_USD_cex_volume_24h   

## Step 2 — Fix timestamps

Right now `collected_at` and `last_updated` are just strings like `'2026-05-22T14:17:13+00:00'`.  
Converting them to proper datetime objects means you can do things like:
- Sort by time
- Calculate how long ago a snapshot was
- Group by hour/day

`utc=True` tells pandas these timestamps are in UTC, which they are (the `+00:00` means UTC).

In [3]:
df = bronze.copy()  # always work on a copy — never mutate the original

timestamp_cols = ['collected_at', 'last_updated', 'quote_USD_last_updated', 'date_added']

for col in timestamp_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

print('Timestamp columns after conversion:')
for col in timestamp_cols:
    if col in df.columns:
        print(f'  {col}: {df[col].dtype}  →  example: {df[col].iloc[0]}')

Timestamp columns after conversion:
  collected_at: datetime64[us, UTC]  →  example: 2026-05-22 14:17:13.742504+00:00
  last_updated: datetime64[us, UTC]  →  example: 2026-05-22 14:15:00+00:00
  quote_USD_last_updated: datetime64[us, UTC]  →  example: 2026-05-22 14:15:00+00:00
  date_added: datetime64[us, UTC]  →  example: 2010-07-13 00:00:00+00:00


## Step 3 — Understand the nulls

**Not all nulls are errors.** Before blindly filling them, understand what each null means:

- `max_supply` is null for ETH, USDT, BNB — these have no hard cap. Null = infinite supply. That's correct.
- `platform_*` columns are null for native coins like BTC, ETH, SOL. They only have values for tokens (e.g. USDT is an ERC-20 on Ethereum). Null = not a token. That's correct.
- `tvl_ratio` is null for almost everything — only DeFi protocols with TVL have this.

Bad nulls (things that should always have a value): none visible in this dataset — the API is reliable.

In [4]:
null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

print('Columns with nulls:')
for col, count in null_summary.items():
    pct = count / len(df) * 100
    print(f'  {col:<45} {count:>3} nulls ({pct:.0f}%)')

Columns with nulls:
  tvl_ratio                                      10 nulls (100%)
  platform                                       10 nulls (100%)
  quote_USD_tvl                                  10 nulls (100%)
  platform_symbol                                 7 nulls (70%)
  platform_id                                     7 nulls (70%)
  platform_name                                   7 nulls (70%)
  platform_slug                                   7 nulls (70%)
  platform_token_address                          7 nulls (70%)
  max_supply                                      6 nulls (60%)
  self_reported_circulating_supply                6 nulls (60%)
  self_reported_market_cap                        6 nulls (60%)


## Step 4 — Flag stablecoins

USDT and USDC are designed to hold $1.00 forever. If you're analyzing price movements or volatility, they'll mess up your results — zero variance, no trend, no momentum.

We don't remove them — we flag them. That way you can filter them in/out depending on what you're analyzing.

In [5]:
STABLECOINS = {'USDT', 'USDC', 'BUSD', 'DAI', 'TUSD', 'FRAX'}

df['is_stablecoin'] = df['symbol'].isin(STABLECOINS)

print('Coin classification:')
print(df[['symbol', 'quote_USD_price', 'is_stablecoin']].to_string(index=False))

Coin classification:
symbol  quote_USD_price  is_stablecoin
   BTC     76976.073298          False
   ETH      2122.466855          False
  USDT         0.998954           True
   BNB       661.340880          False
   XRP         1.356508          False
  USDC         0.999756           True
   SOL        86.912775          False
   TRX         0.360002          False
  DOGE         0.106050          False
  HYPE        60.845617          False


## Step 5 — Add derived columns

### Why log returns instead of percentage change?

The API gives us `percent_change_24h` = percentage change over 24 hours.  
We convert it to **log return**: `ln(1 + pct/100)`

Why? Log returns are:
- **Additive across time**: log return over 2 days = sum of daily log returns
- **Symmetric**: +10% then -10% ≠ 0 in percentage terms, but in log returns it does
- **Better behaved statistically**: closer to normally distributed

This matters when you get to ML — most models assume your features are roughly normal.

In [6]:
# Log returns from the percentage change columns
pct_to_log = {
    'quote_USD_percent_change_1h':  'log_return_1h',
    'quote_USD_percent_change_24h': 'log_return_24h',
    'quote_USD_percent_change_7d':  'log_return_7d',
}

for pct_col, log_col in pct_to_log.items():
    df[log_col] = np.log(1 + df[pct_col] / 100)

# Volume-to-market-cap ratio: high = actively traded relative to its size
df['volume_to_mcap'] = df['quote_USD_volume_24h'] / df['quote_USD_market_cap']

# Supply utilization: how much of max supply is already in circulation
# NaN where max_supply is null (infinite supply coins) — that's correct
df['supply_utilization'] = df['circulating_supply'] / df['max_supply']

# Is this coin a token on another chain (vs a native coin)?
df['is_token'] = df['platform_id'].notna()

print('New derived columns added:')
new_cols = list(pct_to_log.values()) + ['volume_to_mcap', 'supply_utilization', 'is_token', 'is_stablecoin']
print(df[['symbol'] + new_cols].to_string(index=False, float_format='{:.6f}'.format))

New derived columns added:
symbol  log_return_1h  log_return_24h  log_return_7d  volume_to_mcap  supply_utilization  is_token  is_stablecoin
   BTC      -0.005601       -0.000389      -0.028405        0.016473            0.953945     False          False
   ETH      -0.004685        0.003542      -0.044527        0.046969                 NaN     False          False
  USDT       0.000091       -0.000019      -0.000459        0.334958                 NaN      True           True
   BNB       0.001550        0.019084      -0.020026        0.014536            1.000000     False          False
   XRP      -0.007181       -0.001541      -0.056097        0.020538            0.618290     False          False
  USDC       0.000099        0.000075       0.000039        0.142723                 NaN      True           True
   SOL      -0.008283        0.011109      -0.027133        0.069381                 NaN     False          False
   TRX      -0.010071       -0.005135       0.027439        0

## Step 6 — Deduplication

If you accidentally ran the collection notebook twice in quick succession, you'd have near-identical rows for the same coin at almost the same time.

We deduplicate on `(symbol, collected_at)` — one row per coin per snapshot.

In [7]:
before = len(df)
df = df.drop_duplicates(subset=['symbol', 'collected_at'])
after = len(df)

print(f'Rows before dedup: {before}')
print(f'Rows after dedup:  {after}')
print(f'Duplicates removed: {before - after}')

Rows before dedup: 10
Rows after dedup:  10
Duplicates removed: 0


## Step 7 — Quick validation before saving

Before writing Silver data, do a sanity check.  
These are things that should **always** be true. If any fail, something went wrong upstream.

In [8]:
checks = {
    'Price > 0 for all coins':          (df['quote_USD_price'] > 0).all(),
    'Market cap > 0 for all coins':     (df['quote_USD_market_cap'] > 0).all(),
    'Volume >= 0 for all coins':        (df['quote_USD_volume_24h'] >= 0).all(),
    'Rank is 1-10':                     df['cmc_rank'].between(1, 10).all(),
    'No duplicate (symbol, timestamp)': not df.duplicated(subset=['symbol', 'collected_at']).any(),
    'collected_at is not null':         df['collected_at'].notna().all(),
}

all_passed = True
for check, result in checks.items():
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_passed = False
    print(f'  [{status}] {check}')

print()
print('All checks passed!' if all_passed else 'Some checks FAILED — do not save Silver.')

  [PASS] Price > 0 for all coins
  [PASS] Market cap > 0 for all coins
  [PASS] Volume >= 0 for all coins
  [PASS] Rank is 1-10
  [PASS] No duplicate (symbol, timestamp)
  [PASS] collected_at is not null

All checks passed!


## Step 8 — Save Silver

Same append pattern as Bronze — Silver grows every time you run this.

In [9]:
file_exists = silver_path.exists()

df.to_csv(silver_path, mode='a', header=not file_exists, index=False)

total = pd.read_csv(silver_path).shape[0]

print(f'Saved to:   {silver_path}')
print(f'This run:   {len(df)} rows')
print(f'Total rows: {total}')
print(f'Columns:    {df.shape[1]} (Bronze had {bronze.shape[1]})')
print(f'New columns added: {df.shape[1] - bronze.shape[1]}')

Saved to:   C:\Users\User\Documents\Github\CryptoSphere-Analytics-Platform\01-data\silver\crypto_clean.csv
This run:   10 rows
Total rows: 10
Columns:    47 (Bronze had 40)
New columns added: 7


---
## What you just built

```
Bronze (raw CSV)
    ↓  fix types (timestamps → datetime)
    ↓  understand nulls (don't blindly fill them)
    ↓  flag stablecoins
    ↓  add log returns, volume_to_mcap, supply_utilization
    ↓  deduplicate
    ↓  validate
Silver (clean CSV)
```

**What's different between Bronze and Silver:**
- Types are correct (datetime instead of string)
- You have derived columns that are useful for analysis
- It's validated — you know the data is sane

**Next:** Run the collection notebook a few more times (so you have multiple snapshots), then we'll build the Gold layer — aggregations and technical indicators across time.